### This example demonstrates the end-to-end workflow for computing a correlation matrix on a very small synthetic dataset generated by a prompt-AI algorithm.

In [2]:
import time
import matplotlib as plt
import numpy as np
import pandas as pd
from concurrent.futures import ThreadPoolExecutor

#### Single Thread v Multiprocessing

In [3]:
# --- Task 1: Single-threaded CPU Baseline ---
def compute_correlation_serial(X):
    N, T = X.shape
    means = X.mean(axis=1, keepdims=True)
    stds = X.std(axis=1, keepdims=True)
    Z = (X - means) / stds
    C = (Z @ Z.T) / (T - 1)
    return C

# --- Task 2: Multi-core CPU Parallel Version (Threaded) ---
def parallel_cpu_correlation_v2(X, num_workers=4):
    """
    Parallel Engine Implementation: MapReduce Pattern
    ------------------------------------------------
    This function implements a CPU-based parallel correlation engine using 
    a 'Split-Apply-Combine' strategy.

    1. Decomposition (Split): 
       Uses np.array_split to divide the standardized dataset Z into 'num_workers' 
       horizontal chunks. This allows each worker to own a subset of the rows.

    2. Orchestration (Map): 
       Utilizes ThreadPoolExecutor to dispatch each chunk to a separate CPU thread. 
       The 'compute_chunk' function performs the matrix multiplication (chunk @ Z.T) 
       independently for its assigned rows.

    3. Assembly (Combine): 
       The 'list(executor.map(...))' serves as a synchronization barrier, waiting 
       for all threads to finish. Finally, np.vstack 'gathers' the partial results 
       to reconstruct the full N x N correlation matrix.

    Note on Performance:
    While logically parallel, this approach may face overhead from thread 
    management and Python's Global Interpreter Lock (GIL), which can result 
    in a speedup factor < 1.0x for smaller N.
    """
    N, T = X.shape
    # Standardizing (Serial part)
    means = X.mean(axis=1, keepdims=True)
    stds = X.std(axis=1, keepdims=True)
    Z = (X - means) / stds

    # Define the worker operation
    def compute_chunk(chunk):
        return (chunk @ Z.T) / (T - 1)

    # Split dataset into N chunks based on number of workers
    chunks = np.array_split(Z, num_workers)
    
    with ThreadPoolExecutor(max_workers=num_workers) as executor:
        results = list(executor.map(compute_chunk, chunks))
        
    return np.vstack(results)

if __name__ == "__main__":
    # Increasing N and T to see meaningful parallel benefits
    # The project requires scaling from small to very large values [cite: 10]
    N, T = 4000, 4000
    np.random.seed(0)
    X = np.random.randn(N, T)

    print(f"Dataset Size: {N} series x {T} time steps")
    print("-" * 40)

    # 1. Run Serial
    t0 = time.time()
    C_serial = compute_correlation_serial(X)
    t1 = time.time()
    serial_time = t1 - t0
    print(f"Serial CPU Time:   {serial_time:.4f} seconds")

    # 2. Run Parallel
    t2 = time.time()
    C_parallel = parallel_cpu_correlation_v2(X, num_workers=4)
    t3 = time.time()
    parallel_time = t3 - t2
    print(f"Parallel CPU Time: {parallel_time:.4f} seconds")

    # 3. Validation 
    correct = np.allclose(C_serial, C_parallel)
    print(f"Correctness Check: {'PASSED' if correct else 'FAILED'}")
    
    # 4. Speedup Analysis 
    print(f"Speedup Factor:    {serial_time / parallel_time:.2f}x")

Dataset Size: 4000 series x 4000 time steps
----------------------------------------
Serial CPU Time:   0.5840 seconds
Parallel CPU Time: 0.7825 seconds
Correctness Check: PASSED
Speedup Factor:    0.75x



Multi-core CPU Implementation Analysis:
---------------------------------------
This implementation explores the limits and trade-offs of CPU-based parallelism. 

Key Observations:
1. Threading Overhead: For mid-sized matrices (e.g., N=4000), the 'paperwork' 
   of managing a ThreadPool/ProcessPool can exceed the actual computation time, 
   leading to a speedup factor < 1.0x[cite: 18].

2. BLAS Contention: NumPy's matrix multiplication (Z @ Z.T) utilizes highly 
   optimized BLAS libraries which are already multi-threaded. Adding manual 
   parallelism via Python can cause 'thread contention,' where workers compete 
   for the same hardware resources[cite: 12].

3. Problem-Size Regime: This version identifies the regime where manual CPU 
   parallelism is not yet advantageous due to the efficiency of the single-threaded 
   baseline[cite: 11, 18].
